# DADA2: 04 — Evaluation

Three TODOs.

You will conclude the assignment by running a RAG eval.

Three questions to answer with numbers:
- Is the decision model as good a router as the chat model? 
- Does the diagnosis agent beat the ladder, and at what cost?
- which parts of what you built were worth building?

In [1]:
from dotenv import load_dotenv
load_dotenv()

import time
from rag import config, decide, evaluate, retrieval, agent, ladder


## Read this first

This notebook is where the Gemini quota goes. The ladder agent spends about
five chat calls per question and yours about three, and the judge spends one
more on each answer. Fourteen questions through both agents is roughly 150
calls against a limit of 500 a day.

So, you will want to run it once, when notebooks 02 and 03 already pass their final cells. The comparison cells sleep between questions on purpose. Start them and leave them alone.

In [4]:
specific = evaluate.load_evalset("specific")
comparative = evaluate.load_evalset("comparative")
items = specific + comparative

print(f"{len(specific)} specific + {len(comparative)} comparative = {len(items)} labeled questions")
print("reranker:       cohere")
print("chat model:    ", config.CHAT_MODEL)
print("decision model:", config.DECISION_MODEL)


14 specific + 10 comparative = 24 labeled questions
reranker:       cohere
chat model:     gemini-3.5-flash-lite
decision model: clef-flash


## 1. Two routers

### TODO 4.1 — the decision model against the chat model (1 pt)

Every labeled question has a kind: the 14 in `evalset.json` are specific,
the 10 in `evalset_hard.json` are comparative. Ask both models which kind each
one is, and keep the time.

Write the chat-model version yourself. It is a `with_structured_output` call
with a `Literal` field, the same thing you did for `Filter` in notebook 02.
Use `decide.SCOPES` for the descriptions so both models are asked the same
question.

**Steps:**

1. Define a scoper that uses Scope to get structured output from chat model
3. In the loop below, the decision-model line is written. Add the same two lines for the chat model, keyed `"chat"`: time the call, add the seconds, add one to `right` when the choice matches `kind`.


In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from rag.util import retry

# TODO 4.1 — a scoper that uses Scope to get structured output from chat model
class Scope(BaseModel):
    """Which kind of question this is."""
    scope: Literal["specific", "comparative", "figure"] = Field(
        description="; ".join(f"{k}: {v}" for k, v in decide.SCOPES.items()))

def chat_scope(question):
    t = time.time()
    out = retry(scoper.invoke, "Question about SEC 10-K filings: " + question
                + "\nWhat does it ask for?")
    return out.scope, time.time() - t

#TODO 4.1
# your code here
scoper = config.chat().with_structured_output(Scope)

In [5]:
labeled = [(i["question"], "specific") for i in specific] + \
           [(i["question"], "comparative") for i in comparative]

right = {"decision": 0, "chat": 0}
secs = {"decision": 0.0, "chat": 0.0}
for q, kind in labeled:
    t = time.time(); choice, _ = decide.scope(q); secs["decision"] += time.time() - t
    right["decision"] += (choice == kind)
    t = time.time(); choice, _ = chat_scope(q); secs["chat"] += time.time() - t
    right["chat"] += (choice == kind)
    time.sleep(3)

n = len(labeled)
for name in ("decision", "chat"):
    print(f"{name:9} {right[name]}/{n} right, {secs[name]/n:.2f}s per call")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


decision  20/24 right, 0.45s per call
chat      19/24 right, 0.69s per call


NOTE: Accuracy is close, and the chat model is probably not
slower. If you expected the decision model to win on both, write down what it does give you instead: a probability per option, and a call that does not count against the daily limit. Whether those are worth having is the question
the next section answers.

## 2. Two agents

### TODO 4.2 — the ladder against the diagnosis (1 pt)

`evaluate.compare_agents` runs each agent over the questions, has the chat
model judge every answer, and keeps the bill: chat calls, decision calls, and
seconds per question. Both agents expose `run(question)` returning the final
state, which is all it needs.

Fourteen questions is enough. Use eight specific and six comparative so both
kinds are in the mix.

**Steps:** nothing to write. Run the cell once, when both agents pass their final cells. If a 429 cuts it off, score the two agents in separate cells rather than restarting the pair.

This will take a while. Plan accordingly. 

Make sure your rate limit is reset or use a different provider (e.g., Groq, OpenRouter) instead of the Gemini API. 

Let it run and come back to see your results.


In [6]:
# TODO 4.2
subset = specific[:8] + comparative[:6]

rows = evaluate.compare_agents({"ladder": ladder.run, "diagnosis": agent.run}, subset)

  agent         accuracy  chat/q  decision/q    s/q
  --------------------------------------------------


/Users/donnyle/Documents/agentic-ai-course/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ladder            0.86     4.5         0.0   14.0
  diagnosis         0.71     3.4         2.6   13.5


In [7]:
# the judge's reasons, on anything either agent got wrong
_, notes = evaluate.score_agent(agent.run, subset[:4])
for note in notes:
    mark = "ok " if note["correct"] else "WRONG"
    print(f"[{mark}] {note['question'][:66]}  ({note['chat_calls']} chat, {note['decision_calls']} decision)")
    print(f"        {note['reason'][:150]}\n")

[ok ] What does MU state that the preparation of financial statements an  (2 chat, 2 decision)
        CORRECT
The provided answer accurately identifies management's responsibility regarding estimates and judgments in financial statement preparation und

[ok ] What termination fee is AMD required to pay ZT Systems under certa  (2 chat, 2 decision)
        CORRECT
The answer correctly identifies the $300 million termination fee related to regulatory approvals for the ZT Systems acquisition as stated in t

[ok ] What factors does NVIDIA believe are the principal competitive fac  (2 chat, 2 decision)
        CORRECT
The provided answer accurately lists the principal competitive factors for NVIDIA for fiscal year 2025 as stated in the filing.

[ok ] What competitive pressures from Intel and Nvidia did AMD face in f  (2 chat, 2 decision)
        CORRECT
The answer accurately and comprehensively identifies the competitive pressures AMD faced from Intel and Nvidia in fiscal year 2025 as suppo

If the diagnosis agent got something wrong that the ladder got right, find
out which half failed. Run `decide.scope` and `decide.diagnose` on that
question by hand and look at the probabilities. A wrong route and a wrong
verdict are different bugs, and the fix for one is a threshold while the fix
for the other is a search strategy.

### TODO 4.3 — write it up (2 pts)

In the cell below, in your own words. Four or five short paragraphs.

**Conditions.** Reranker, chat model, decision model, k, chunk size. Enough
for someone to re-run you.

**The routers.** Which was more accurate, which was faster, and by how much.
Then say what the decision model is for, given those numbers.

**The agents.** Accuracy, chat calls per question, seconds per question, for
both. Name the question where they differed most and say why.

**One thing that did not help.** Something you built that the numbers say you
could remove. Say what the mechanism is, not that the model is better at it.

**What you would ship**, and what you would want to know about your users'
questions first.

Two things that cost marks: an improvement the table does not show, and a
difference explained by "the model is better" when the reason is in your
code.

**Steps:** write it as prose, four or five short paragraphs. The bold headings in the cell below are the order; replace the cell's text under each. Every number you quote should be in a table above.


### My write-up

**Conditions.** I used Gemini gemini-3.5-flash-lite for chat, Cloudflare clef-flash for decisions, and Cohere rerank-v3.5 for reranking. The embeddings were Qwen3 0.6B through Cloudflare, with a chunk size of 1,200 characters and an overlap of 150. The reranking pool held 10 passages, and the RRF constant was 60. I tested retrieval on six specific questions at k=5 and five comparative questions at k=10. The router comparison used all 24 questions while the agent comparison used eight specific and six comparative questions at k=5.

**The routers.** The decision router got 20 out of 24 questions right compared to 19 for the chat router. It was also faster, averaging 0.45 seconds per call compared to 0.69 seconds. I think this is a small advantage, especially since the accuracy difference comes down to one question. What makes the decision model useful to me is that it gives probabilities that the code can actually use. If it is unsure, the agent can fall back to another strategy. It also avoids spending Gemini calls just to choose a search strategy. I would still want more questions before saying it is consistently the better router.

**The agents.** The ladder agent had an accuracy of 0.86, compared to 0.71 for the diagnosis agent. The diagnosis agent used fewer chat calls, averaging 3.4 per question compared to 4.5, but it also used 2.6 decision calls. Their times were close, at 13.5 seconds for diagnosis and 14.0 for the ladder agent. I think this was the clearest tradeoff in the results: saving chat calls did not mean getting more questions right. The AMD dress-code example stood out because the diagnosis agent stopped after its first search with an absent verdict, while the ladder agent took three attempts and nine chat calls to reach a similar answer. That shows where the diagnosis agent saved work even though it was less accurate overall. The saved comparison only gives aggregate scores so I can't tell which question had the biggest accuracy disagreement.

**What did not help.** Adding multi-query retrieval after filtering did not help on my six specific questions. Recall@5 stayed at 1.00, while MRR dropped from 0.722 to 0.694. Adding reranking brought MRR back to 0.722, which was the same as filtering alone. Filtering by company and year had already found the relevant files. The extra versions of the query seem to have added more noise to the ranking without bringing in any additional relevant files. Since this also took another chat call, I would just use filtering for these questions. However, this does not mean reranking was useless overall: without filtering, it increased the baseline MRR from 0.325 to 0.583. Hybrid retrieval also did well on the specific questions, with an MRR of 0.783, so my results did not match every example in the starter.

**What I would ship.** Based on this run, I would ship the ladder agent first because I care more about getting the answer right than saving roughly one chat call per question, especially when the time difference was so small. Cost would still matter if a lot of people were using the app, since those extra chat calls would add up and use more of the available quota. The diagnosis agent used fewer chat calls, but it also made decision-model calls, so I would compare token usage and the cost of both models before deciding which is actually cheaper. For now, I think the higher accuracy is worth the extra chat usage. I still like the diagnosis agent's ability to stop on unsupported questions and plan missing searches, but I would want to understand its mistakes before choosing it over the ladder agent. I would also ask what people actually use the app for. On the five comparative questions, filtering lowered recall@10 from 0.83 to 0.67, so a strategy that works well for a named company and year may be a poor fit for broader questions. Before shipping, I would check more examples of each kind and review the answers and citations myself, since the accuracy scores also depend on a model judging the answers.

**Copy your write-up to README.md.**